In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('../results/routing_dataset.csv')
features = ['degree', 'betweenness', 'closeness', 'traffic_load']
X = df[features]
y = df['optimal_next_hop']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 3998
Test size: 1000


In [2]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"RF Accuracy: {acc*100:.2f}%")

# Feature importance
importances = pd.Series(rf.feature_importances_, index=features)
print("\nFeature Importances:")
print(importances.sort_values(ascending=False))

RF Accuracy: 32.90%

Feature Importances:
traffic_load    0.603413
betweenness     0.208366
closeness       0.133939
degree          0.054282
dtype: float64


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx

np.random.seed(42)
G = nx.erdos_renyi_graph(50, 0.15, seed=42)
for u, v in G.edges():
    G[u][v]['latency'] = round(np.random.uniform(1, 10), 2)

deg = nx.degree_centrality(G)
bet = nx.betweenness_centrality(G)
clo = nx.closeness_centrality(G)
nodes = list(G.nodes())

results = []
for _ in range(1000):
    src = np.random.choice(nodes)
    dst = np.random.choice([n for n in nodes if n != src])
    path = [src]
    current = src
    visited = set([src])
    delivered = False
    delay = 0

    for _ in range(50):
        neighbors = [n for n in G.neighbors(current) if n not in visited]
        if not neighbors:
            break
        feat = pd.DataFrame([{
            'degree': deg[current],
            'betweenness': bet[current],
            'closeness': clo[current],
            'traffic_load': np.random.uniform(0, 1)
        }])
        next_hop = rf.predict(feat)[0]
        if next_hop not in neighbors:
            next_hop = neighbors[0]
        delay += G[current][next_hop]['latency']
        path.append(next_hop)
        visited.add(next_hop)
        if next_hop == dst:
            delivered = True
            break
        current = next_hop

    results.append({'delivered': delivered,
                    'hops': len(path)-1, 'delay': delay})

res_df = pd.DataFrame(results)
pdr  = round(res_df['delivered'].mean()*100, 2)
avgd = round(res_df[res_df['delivered']]['delay'].mean(), 2)
avgh = round(res_df[res_df['delivered']]['hops'].mean(), 2)

print(f"RF Routing — PDR: {pdr}%  |  Avg Delay: {avgd} ms  |  Avg Hops: {avgh}")
res_df.to_csv('../results/ml_baseline.csv', index=False)

In [ ]:
print(f"RF Routing — PDR: {pdr}%  |  Avg Delay: {avgd} ms  |  Avg Hops: {avgh}")
print("Total rows in ml_baseline.csv:", len(res_df))

In [ ]:
import pandas as pd
res_df = pd.read_csv('../results/ml_baseline.csv')
pdr = round(res_df['delivered'].mean()*100, 2)
avgd = round(res_df[res_df['delivered']]['delay'].mean(), 2)
avgh = round(res_df[res_df['delivered']]['hops'].mean(), 2)
print(f"RF Routing — PDR: {pdr}%  |  Avg Delay: {avgd} ms  |  Avg Hops: {avgh}")